<a href="https://colab.research.google.com/github/CPTR295/Learning-Langchain/blob/main/Advanced_Indexing_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-openai langchain-community langchain-chroma chromadb lxml html2text lark

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51

In [2]:
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")

In [3]:
import requests
from langchain_core.embeddings import Embeddings
from google.colab import userdata

api_key = userdata.get("OPENROUTER_API_KEY")

class OpenRouterNvidiaEmbeddings(Embeddings):
    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://openrouter.ai/api/v1/embeddings"
        self.model = "nvidia/llama-nemotron-embed-vl-1b-v2:free"

    def _embed(self, texts, input_type):
        response = requests.post(
            self.url,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": self.model,
                "input": [
                    {
                        "content": [
                            {"type": "text", "text": text}
                        ]
                    }
                    for text in texts
                ],
                "input_type": input_type,
                "encoding_format": "float",
            },
            timeout=120,
        )

        response.raise_for_status()
        data = response.json()["data"]
        data.sort(key=lambda item: item["index"])

        return [item["embedding"] for item in data]

    def embed_documents(self, texts):
        return self._embed(texts, "passage")

    def embed_query(self, text):
        return self._embed([text], "query")[0]


embeddings_model = OpenRouterNvidiaEmbeddings(api_key)

Preparing the Chroma DB collections

In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

In [6]:
corwnall_granular_collection = Chroma(
    collection_name="cornwall_granular",
    embedding_function=embeddings_model
) #New Chorma DB collection
corwnall_granular_collection.reset_collection() #Reset the collection

In [7]:
corwnall_coarse_collection = Chroma(
    collection_name="cornwall_coarse",
    embedding_function=embeddings_model
) #New Chorma DB collection
corwnall_coarse_collection.reset_collection() #Reset the collection

Loading the HTML content with the AsyncHtmlLoader

In [8]:
from langchain_community.document_loaders import AsyncHtmlLoader

In [26]:
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"

In [79]:
destination_url ="https://www.visitbritain.com/en/destinations/england/cornwall-and-isles-scilly"

In [80]:
html_loader = AsyncHtmlLoader(destination_url)

In [81]:
docs = html_loader.load()

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.38it/s]


In [82]:
len(docs)

1

In [83]:
docs

[Document(metadata={'source': 'https://www.visitbritain.com/en/destinations/england/cornwall-and-isles-scilly', 'title': 'Cornwall & Isles of Scilly Places To Visit | VisitBritain', 'description': 'Heading to Cornwall or the Isles of Scilly? View our official guide to find the best beaches, attractions, and places to visit.', 'language': 'en-gb'}, page_content='<!DOCTYPE html><html lang="en-gb" dir="ltr" prefix="og: https://ogp.me/ns#" class="theme--vb no-js u-h-full"><head><script>\nwindow.dataLayer = window.dataLayer || [];\nfunction gtag(){dataLayer.push(arguments);}\ngtag(\'consent\', \'default\', {\n\'ad_storage\': \'denied\',\n\'ad_user_data\': \'denied\',\n\'ad_personalization\': \'denied\',\n\'analytics_storage\': \'denied\',\n\'personalization_storage\': \'denied\',\n\'wait_for_update\': 500\n});\n</script><script>\n(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({\'gtm.start\':\nnew Date().getTime(),event:\'gtm.js\'});var f=d.getElementsByTagName(s)[0],\nj=d.createElement(s),dl=

Splitting into granular chunks with the HTMLSectionSplitter

In [84]:
from langchain_text_splitters import HTMLSectionSplitter

In [85]:
headers_to_split_on = [("h1","Header 1"), ("h2","Header 2"), ("h3", "Header 3")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on
)

In [86]:
def split_docs_into_granular_chunks(docs):
  all_chunks =[]
  for doc in docs:
    html_string = doc.page_content
    temp_chunks = html_section_splitter.split_text(html_string) #Extract HTML text from the document
    all_chunks.extend(temp_chunks)
  return all_chunks

In [87]:
granular_chunks = split_docs_into_granular_chunks(docs)

In [88]:
len(granular_chunks)

175

Ingesting granular chunks

In [89]:
corwnall_granular_collection.add_documents(documents=granular_chunks)

['5b45465c-979f-4548-a7e4-1139f594ef05',
 '6cdfe28f-bf1e-4b20-a858-c3d3e6030d40',
 '5bcb9ce6-9974-4540-a515-1345c485607a',
 '309d47eb-734a-4437-b534-11d9f5fdcdc9',
 '7c6a1dc5-26d0-4f5f-a566-92d0cb2a8f37',
 '95276d0d-afea-4b1d-bc48-01df2179beb0',
 'c44e1a96-5712-4e0b-ba47-a2e7f35d1159',
 '19b40929-3fc5-4a17-a31b-06779429f4b3',
 '2b215d67-34a3-47fe-b79b-3ef721bed45f',
 '6ec0b914-c433-4f5a-ab8a-b06a5c1be743',
 'f66b8dec-e373-4cc2-921e-7a4fd9d532a5',
 '4eb6aca9-648a-4b5c-a26c-49c60c2b580e',
 '69589cf5-7154-4286-b2d4-e9756c021c76',
 'fce8c951-1a60-42f2-bd50-6d9f93396d30',
 '7bdda91c-facf-4399-b22d-e025f44fddcc',
 '127363d1-fd09-47ef-8285-f76c7bb83d82',
 'ec7cdbeb-dc6b-40d0-b565-03f3923f60cc',
 '271564e5-5308-4b63-85f2-f2b36284af66',
 '9777d46b-690b-4023-9ac3-9827679740e0',
 '7db4a302-1295-4b44-9475-648b5716bbc0',
 'cc734750-3ea0-4f99-a3e0-003deea9a1de',
 '21c34991-fa66-4bd4-af8d-5cd941640f3f',
 'cbd994ee-dc0d-48bd-8441-214c5ecc40a3',
 '29bd7042-fb00-422c-b3a3-0994b17a810b',
 'd28250be-abe6-

Searching granular chunks

In [90]:
results = corwnall_granular_collection.similarity_search(
    query="Events or festivals in Corwnall",k=3
)
for doc in results:
  print(doc)

page_content='Golowan Festival  
  View on map  
  June 2027  
 
  In the height of June, the streets of Penzance fill with live music, parades and traditional dancing. 
  More about Golowan Festival  (opens in a new tab) 
 
 
 
 
 Photo by:  Alamy 
 
 
 Location:  Penzance, England 
 
 
 
 [if IE 9]><video style="display: none;"><![endif] [if IE 9]></video><![endif] 
 
 
 
 
 
 
 schema.org place microdata   Golowan Festival' metadata={'Header 3': 'Golowan Festival'}
page_content='Golowan Festival  
  View on map  
  June 2027  
 
  In the height of June, the streets of Penzance fill with live music, parades and traditional dancing. 
  More about Golowan Festival  (opens in a new tab) 
 
 
 
 
 Photo by:  Alamy 
 
 
 Location:  Penzance, England 
 
 
 
 [if IE 9]><video style="display: none;"><![endif] [if IE 9]></video><![endif] 
 
 
 
 
 
 
 schema.org place microdata   Golowan Festival' metadata={'Header 3': 'Golowan Festival'}
page_content='Cornwall Folk Festival  
  View on map  

Splitting into coarse chunks with the RecursiveCharacterTextSplitter

In [92]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [93]:
html2text_transformer = Html2TextTransformer()

In [94]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000,chunk_overlap=500)

In [96]:
def split_docs_into_coarse_chunks(docs):
  text_docs = html2text_transformer.transform_documents(docs)
  coarse_chunks = text_splitter.split_documents(text_docs)
  return coarse_chunks

In [97]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

Ingesting Coarse Chunks

In [99]:
corwnall_coarse_collection.add_documents(documents=coarse_chunks)

['b8b438db-13a2-4f32-9b7d-50e5280092a4',
 '8c9484da-06eb-41f2-8335-f78f05c3edc8',
 '8c302acf-1fa6-475a-943e-7543d3648eba',
 '36803bdb-8bbb-48be-86a8-e74abbd4780c',
 'ec16f5f6-cfe0-448f-9153-5377bc814fde',
 'a6521963-e96c-45b7-aa07-0bb0bf52b2ed',
 '0ef1618c-d886-4ff9-9bc9-555bacdba22c',
 '86081310-a8b6-4f06-a2f2-2e10eee511cd',
 '132cfe99-34c0-4615-b321-ef14bcd9ef03',
 '906ac576-3f5f-478a-ba2c-e91f3759ddb3',
 '3bc4a315-2d9e-427d-9720-b902faa058af',
 '602b7983-169f-4dda-ae27-443af10280b0',
 '68c39835-f89d-4ad2-9457-4604ad779577',
 'bfe23393-9acd-4ef4-a93b-4dae2ba67471',
 '1fc5e418-3c72-4833-8c6f-6efda5d26fde',
 '1d541c4e-e093-4510-b9c1-69ea792f845b',
 'eb1c2343-d940-4214-84d8-f24102e513a8',
 '5a7532db-59a5-442d-9bf0-471a28f92414',
 '4121215b-a4e9-40a6-ad7a-137f81f5e887',
 '42fac721-c77a-4427-9da8-de09b7546ed3',
 'a54da205-057d-4359-8f56-fc7585774740']

Searching  Coarse chunks

In [100]:
results = corwnall_coarse_collection.similarity_search(
    query="Events or festivals in Corwnall",k=3
)
for doc in results:
  print(doc)

page_content='Discover Cornwall Pride (opens in a new tab)

Photo by: Cornwall Pride / Kai Greet Photography

Location: Penzance, England

Cornwall Pride

###  Rattler Fest

View on map

7th - 8th May 2027

Amazing music, great atmosphere and free-flowing cider: is there a better
recipe for a summer festival?

Explore Rattler Fest (opens in a new tab)

Photo by: Rattler Fest

Location: Truro, England

Rattler Fest

###  Golowan Festival

View on map

June 2027

In the height of June, the streets of Penzance fill with live music, parades
and traditional dancing.

More about Golowan Festival (opens in a new tab)

Photo by: Alamy

Location: Penzance, England

Golowan Festival

###  St Michaels Resort

View on map

Look forward to a sumptuous spa, top-quality restaurant, and comfortable
beach-side rooms.

Explore St Michaels Resort (opens in a new tab)

Photo by: St Michaels Resort, Cornwall

Location: Falmouth, England

St Michaels Resort

###  Holywell

View on map

Surf’s up at this bay

Embedding Strategy

Embedding child chunks with ParentDocumenRetriever

In [101]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

Parent Document Retriever

In [103]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=5000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

child_chunks_collection = Chroma(collection_name="corwnall_child_chunks",embedding_function=embeddings_model)
child_chunks_collection.reset_collection()

doc_store = InMemoryStore() #Document store to host parent coarse chunks

parent_doc_retriever = ParentDocumentRetriever( #Retriever to link parent coarse chunks to granular chunks
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)


Ingestion the content

In [105]:
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()
text_docs = html2text_transformer.transform_documents(docs)
parent_doc_retriever.add_documents(coarse_chunks)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.55it/s]


In [106]:
list(doc_store.yield_keys())

['e87d3b27-b402-43fd-8f13-2adc13a75675',
 '26e8eb03-0fa7-4ee3-9eb2-62dfe04b3c9e',
 'c6898e71-7617-4412-bfb8-2efbdab42145',
 '27f14623-b589-441b-bce8-2d686007a59d',
 '42a9d134-f8a9-4139-8bf3-93a6ca2a5887',
 '84a96aec-1e6a-40f4-98e6-d79dcd8d0089',
 '8e6a4263-1981-4cca-824c-c23f99c13312',
 '5507bb2b-9c5f-4f40-9f5d-9abbfa6b83fa',
 'ec0cb3ec-1525-4edb-8af4-8cbbe8ee49d4',
 '8258ffef-e784-402b-bbab-e914823d89b8',
 'f90a9890-16d9-4fdd-8e8c-7b4b1d50882e',
 '91cfe1eb-5ec6-4454-8b25-c4219ed45cde',
 '5f841e72-e34e-4a05-b25c-ded2801f317a',
 '06d6c3c2-8f12-4f05-9ffc-7c40e04b6f9b',
 '7634b1ea-bc9c-4695-9581-0b6ccb8c272c',
 '6cda6d37-0855-4ff6-bc4b-ba7ce460ffaa',
 '39a4af01-04b2-4453-a0c8-2715c4793518',
 'fc13975d-f9fd-4241-9c36-a923182c311f',
 'f3d497ea-260b-4cc1-9b49-b762790235b6',
 'af0d11c6-8936-4fdb-ab8c-f0b31b735d3d',
 '410b1d54-e807-4c4c-883f-f44632fdd0f8']

Performing a search on granular information

In [108]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [109]:
len(retrieved_docs)

3

In [110]:
retrieved_docs[0]

Document(metadata={'source': 'https://www.visitbritain.com/en/destinations/england/cornwall-and-isles-scilly', 'title': 'Cornwall & Isles of Scilly Places To Visit | VisitBritain', 'description': 'Heading to Cornwall or the Isles of Scilly? View our official guide to find the best beaches, attractions, and places to visit.', 'language': 'en-gb'}, page_content='Explore Newlyn Art Gallery & The Exchange (opens in a new tab)\n\nPhoto by: Newlyn Art Gallery\n\nLocation: Penzance, England\n\nNewlyn Art Gallery & The Exchange\n\n###  Trevibban Mill\n\nView on map\n\nFor a taste of wild Cornwall, look to this family-run vineyard, farm and\norchards.\n\nGet to know Trevibban Mill (opens in a new tab)\n\nPhoto by: Trevibban Mill, Cornwall\n\nLocation: Padstow, England\n\nTrevibban Mill\n\n###  Philleigh Way Cornish Cookery School\n\nView on map\n\nStart the day at a farmhouse-style cookery school, where you’ll mix up a\nCornish classic, the pasty.\n\nTry the Philleigh Way Cornish Cookery School

Comparing with direct semantic search on child chunks

In [111]:
child_docs_only = child_chunks_collection.similarity_search(
    query="Cornwall Ranger",k=3
)

In [112]:
child_docs_only[0]

Document(id='ab6577fe-8046-4756-b277-3d86d24bd0f0', metadata={'description': 'Heading to Cornwall or the Isles of Scilly? View our official guide to find the best beaches, attractions, and places to visit.', 'title': 'Cornwall & Isles of Scilly Places To Visit | VisitBritain', 'source': 'https://www.visitbritain.com/en/destinations/england/cornwall-and-isles-scilly', 'language': 'en-gb', 'doc_id': '5507bb2b-9c5f-4f40-9f5d-9abbfa6b83fa'}, page_content='Photo by: National Trust Images / Hugh Mothersole\n\nLocation: Botallack, England\n\nBotallack Mine\n\n###  Cornwall Pride\n\nView on map\n\nFebruary - October 2026\n\nCelebrate the LGBTQIA+ community with colourful street parades, epic club\nnights and craft markets.\n\nDiscover Cornwall Pride (opens in a new tab)\n\nPhoto by: Cornwall Pride / Kai Greet Photography\n\nLocation: Penzance, England\n\nCornwall Pride\n\n###  Rattler Fest\n\nView on map\n\n7th - 8th May 2027')

Embedding child chunks with MultiVectorRetriever

In [113]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
import uuid

Setting up the Multi vector retriever

In [114]:
child_chunks_collection.reset_collection()
doc_byte_store = InMemoryStore()
doc_key = "doc_id"
multi_vector_retriever = MultiVectorRetriever(
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store)